# Fama-French Five-Factor Benchmark
Cross-sectional equity return prediction in the Emerging Markets universe. The five Fama-French factor proxies used are book-to-market (value), operating profitability divided by book equity (profitability), asset growth (investment), past 12-month returns (momentum), and market equity (size).

Three families of portfolios are computed. First, a value-weighted market portfolio is reported as the long-only reference against MSCI EM Gross. Second, single-factor quintile portfolios are constructed for each of the five factors in both long-short and long-only form. Third, the Fama-MacBeth cross-sectional regression is estimated on the same five factor proxies, and an out-of-sample predictive portfolio is constructed from the expanding-window slope coefficients, again in both long-short and long-only form.

All portfolios use the same six-month rebalancing schedule, 25 basis-point transaction cost, and volatility overlay as the Dual Path Portfolio Transformer. Both raw and volatility-targeted Sharpe ratios are saved.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')

In [ ]:
# configuration

data_path = Path('../data/Global_Factor_EM.parquet')
results_dir = Path('../results/em/ff_benchmark')
results_dir.mkdir(parents = True, exist_ok = True)

# jkp column names for the five fama-french factor proxies
char_map = {
    'value': 'be_me',
    'profitability': 'ope_be',
    'investment': 'at_gr1',
    'momentum': 'ret_12_1',
    'size': 'me',
}
fm_chars = list(char_map.values())

ret_col = 'ret_exc_lead1m'
rebalance_freq = 6
tc_bps = 25
min_stocks = 30
ret_clip_low = -1.0
ret_clip_high = 1.0

target_vol = 0.10
vol_lookback = 6
max_leverage_ls = 3.0
max_leverage_lo = 3.0

id_cols = ['id', 'eom', 'excntry', ret_col, 'me']


### Load and process

In [ ]:

schema = pq.read_schema(data_path)
all_col_names = schema.names

factor_cols = list(char_map.values())
fm_available = [c for c in fm_chars if c in all_col_names]
all_chars = list(dict.fromkeys(factor_cols + fm_available))
needed = list(dict.fromkeys([c for c in id_cols + all_chars if c in all_col_names]))

df = pd.read_parquet(data_path, columns = needed)
df['eom'] = pd.to_datetime(df['eom'])

print(f'rows loaded, {df.shape[0]:,}')
print(f'columns loaded, {df.shape[1]}')
print(f'date range, {df["eom"].min().date()} to {df["eom"].max().date()}')

for col in all_chars:
    if col in df.columns and df[col].dtype == np.float64:
        df[col] = df[col].astype(np.float32)
if 'me' in df.columns and df['me'].dtype == np.float64:
    df['me'] = df['me'].astype(np.float32)

df[ret_col] = df[ret_col].clip(lower = ret_clip_low, upper = ret_clip_high)

### Build monthly cross-sections

In [ ]:

sorted_eoms = sorted(df['eom'].unique())
all_months = {}

for eom in sorted_eoms:
    month = df[df['eom'] == eom].copy()
    month = month[month[ret_col].notna()]
    if len(month) < min_stocks:
        continue

    entry = {
        'ids': month['id'].values,
        'r': month[ret_col].values.astype(np.float64),
        'me': (month['me'].values.astype(np.float64)
                if 'me' in month.columns else np.ones(len(month))),
    }

    for fname, cname in char_map.items():
        entry[fname] = (month[cname].values.astype(np.float64)
                        if cname in month.columns else np.full(len(month), np.nan))

    fm_vals = {}
    for cname in fm_available:
        vals = (month[cname].values.astype(np.float64)
                if cname in month.columns else np.full(len(month), np.nan))
        valid = np.isfinite(vals)
        ranked = np.zeros(len(month))
        if valid.sum() > 5:
            ranked[valid] = pd.Series(vals[valid]).rank(pct = True).values - 0.5
        fm_vals[cname] = ranked

    entry['fm_x'] = np.column_stack([fm_vals[c] for c in fm_available])
    all_months[eom] = entry

sorted_dates = sorted(all_months.keys())
print(f'processed months, {len(sorted_dates)}')

In [ ]:
def portfolio_metrics(rets):
    rets = np.array(rets, dtype = np.float64)
    if len(rets) == 0:
        return {}
    tw = float((1.0 + rets).prod())
    ann_ret = -1.0 if tw <= 0 else float(tw ** (12.0 / len(rets)) - 1.0)
    ann_vol = float(rets.std() * np.sqrt(12.0))
    sharpe = ann_ret / max(ann_vol, 1e-8)
    se = float(np.sqrt((1.0 + 0.5 * sharpe ** 2) / len(rets)))
    cw = np.cumprod(1.0 + rets)
    pk = np.maximum.accumulate(cw)
    max_dd = float(((pk - cw) / pk).max()) if len(cw) > 0 else 0.0
    return {
        'ann_ret': ann_ret, 'ann_vol': ann_vol, 'sharpe': sharpe,
        'se_sharpe': se, 'max_dd': max_dd, 'n_obs': len(rets),
    }


def apply_vol_target(monthly_rets, rebalance_indices, target_vol, vol_lookback, max_leverage):
    scaled = np.array(monthly_rets, dtype = np.float64)
    n = len(monthly_rets)
    n_rb = len(rebalance_indices)
    period_rets = []
    for i in range(1, n_rb):
        window = np.array(monthly_rets[rebalance_indices[i - 1]:rebalance_indices[i]])
        period_rets.append(float(np.prod(1.0 + window) - 1.0))
    for i in range(n_rb):
        if i < vol_lookback:
            continue
        trailing = np.array(period_rets[max(0, i - vol_lookback):i])
        if len(trailing) < 2:
            continue
        sigma_ann = float(trailing.std() * np.sqrt(12.0 / rebalance_freq))
        lev = float(np.clip(target_vol / max(sigma_ann, 1e-8), 1.0 / max_leverage, max_leverage))
        next_rb = rebalance_indices[i + 1] if i + 1 < n_rb else n
        scaled[rebalance_indices[i]:next_rb] = (
            np.array(monthly_rets[rebalance_indices[i]:next_rb]) * lev
        )
    return scaled

### Market portfolio (value-weighted, long-only by construction)

In [ ]:

market_rets = []
for eom in sorted_dates:
    m = all_months[eom]
    valid = np.isfinite(m['me']) & (m['me'] > 0) & np.isfinite(m['r'])
    if valid.sum() < 5:
        continue
    w = m['me'][valid] / m['me'][valid].sum()
    market_rets.append(float((w * m['r'][valid]).sum()))

market_rets = np.array(market_rets)
mkt_m = portfolio_metrics(market_rets)

print(f'market months,{len(market_rets)}')
print(f'market sharpe,{mkt_m["sharpe"]:.4f}')
print(f'market ann_ret,{mkt_m["ann_ret"] * 100:.2f}%')
print(f'market ann_vol,{mkt_m["ann_vol"] * 100:.2f}%')

### Quintile factor portfolios (long-short and long-only)

In [ ]:
def quintile_factor(factor_name, reverse = False):
    """
    Build both long-short and long-only quintile portfolios from a single factor.

    Long-short, top quintile (or bottom when reverse) minus opposite quintile.
    Long-only,  top quintile (or bottom when reverse) only.
    Turnover is computed per leg with the long-only counting only the long set.
    """
    rset = set(sorted_dates[::rebalance_freq])
    ls_rets, ls_rb_indices = [], []
    lo_rets, lo_rb_indices = [], []
    li_ids, si_ids = set(), set()
    prev_li, prev_si = set(), set()

    for eom in sorted_dates:
        m = all_months[eom]
        ids = m['ids']
        r = m['r']
        char_vals = m.get(factor_name)
        if char_vals is None:
            continue

        ls_tcv = 0.0
        lo_tcv = 0.0

        if eom in rset:
            ls_rb_indices.append(len(ls_rets))
            lo_rb_indices.append(len(lo_rets))
            valid = np.isfinite(char_vals)
            if valid.sum() < 10:
                continue
            vi = ids[valid]
            vc = char_vals[valid]
            nq = max(1, int(len(vi) * 0.20))
            so = np.argsort(vc)
            if reverse:
                li_ids = set(vi[so[:nq]].tolist())
                si_ids = set(vi[so[::-1][:nq]].tolist())
            else:
                li_ids = set(vi[so[::-1][:nq]].tolist())
                si_ids = set(vi[so[:nq]].tolist())

            ls_to = (
                len(li_ids - prev_li) + len(prev_li - li_ids)
                + len(si_ids - prev_si) + len(prev_si - si_ids)
            ) / max(nq, 1)
            ls_tcv = ls_to * tc_bps / 10000.0

            lo_to = (
                len(li_ids - prev_li) + len(prev_li - li_ids)
            ) / max(nq, 1)
            lo_tcv = lo_to * tc_bps / 10000.0

            prev_li = li_ids
            prev_si = si_ids

        if not li_ids:
            continue
        il = ids.tolist()
        lr = r[np.array([i in li_ids for i in il])]
        sr = r[np.array([i in si_ids for i in il])]
        lr_mean = float(lr.mean()) if len(lr) > 0 else 0.0
        sr_mean = float(sr.mean()) if len(sr) > 0 else 0.0
        ls_rets.append(lr_mean - sr_mean - ls_tcv)
        lo_rets.append(lr_mean - lo_tcv)

    return {
        'long_short': {'returns': np.array(ls_rets), 'rb_indices': ls_rb_indices},
        'long_only': {'returns': np.array(lo_rets), 'rb_indices': lo_rb_indices},
    }

In [ ]:
# run the five factor portfolios and report per-factor metrics

factor_defs = [
    ('value', False), ('momentum', False), ('profitability', False),
    ('investment', True), ('size', True),
]

factor_results = {}
rows_for_table = []

for fname, rev in factor_defs:
    sim = quintile_factor(fname, reverse = rev)
    ls, lo = sim['long_short'], sim['long_only']
    if len(ls['returns']) == 0:
        print(f'factor, {fname}, no data')
        continue
    ls_scaled = apply_vol_target(ls['returns'], ls['rb_indices'], target_vol, vol_lookback, max_leverage_ls)
    lo_scaled = apply_vol_target(lo['returns'], lo['rb_indices'], target_vol, vol_lookback, max_leverage_lo)
    factor_results[fname] = {
        'returns_ls_raw': ls['returns'], 'returns_ls_scaled': ls_scaled,
        'returns_lo_raw': lo['returns'], 'returns_lo_scaled': lo_scaled,
        'metrics_ls_raw': portfolio_metrics(ls['returns']),
        'metrics_ls_scaled': portfolio_metrics(ls_scaled),
        'metrics_lo_raw': portfolio_metrics(lo['returns']),
        'metrics_lo_scaled': portfolio_metrics(lo_scaled),
    }
    mls = factor_results[fname]['metrics_ls_scaled']
    mlo = factor_results[fname]['metrics_lo_scaled']
    rows_for_table.append({
        'factor': fname,
        'ls_sharpe': round(mls['sharpe'], 4),
        'ls_ann_ret': round(mls['ann_ret'] * 100, 2),
        'ls_ann_vol': round(mls['ann_vol'] * 100, 2),
        'lo_sharpe': round(mlo['sharpe'], 4),
        'lo_ann_ret': round(mlo['ann_ret'] * 100, 2),
        'lo_ann_vol': round(mlo['ann_vol'] * 100, 2),
    })

factor_table = pd.DataFrame(rows_for_table)
print('Quintile Factor Portfolios, EM Universe, vol-targeted')
print(factor_table.to_string(index = False))

### Fama-macbeth cross-sectional regression

In [ ]:

fm_betas = []
fm_dates_used = []

for eom in sorted_dates:
    m = all_months[eom]
    x = m['fm_x']
    r = m['r']
    valid = np.isfinite(r)
    for j in range(x.shape[1]):
        valid = valid & np.isfinite(x[:, j])
    if valid.sum() < len(fm_available) + 5:
        continue
    x_aug = np.column_stack([np.ones(valid.sum()), x[valid]])
    try:
        beta = np.linalg.lstsq(x_aug, r[valid], rcond = None)[0]
        fm_betas.append(beta)
        fm_dates_used.append(eom)
    except np.linalg.LinAlgError:
        continue

fm_betas = np.array(fm_betas)
n_months_fm = len(fm_betas)
fm_mean = fm_betas.mean(axis = 0)
fm_se = fm_betas.std(axis = 0) / np.sqrt(n_months_fm)
fm_tstat = fm_mean / np.maximum(fm_se, 1e-10)

coef_names = ['intercept'] + fm_available
fm_results_table = []
for i, name in enumerate(coef_names):
    p_val = 2.0 * (1.0 - stats.t.cdf(abs(fm_tstat[i]), df = n_months_fm - 1))
    sig = '***' if p_val < 0.01 else '**' if p_val < 0.05 else '*' if p_val < 0.10 else ''
    fm_results_table.append({
        'variable': name,
        'mean_coef': round(float(fm_mean[i]), 5),
        'se': round(float(fm_se[i]), 5),
        't_stat': round(float(fm_tstat[i]), 4),
        'p_value': round(float(p_val), 4),
        'sig': sig,
    })

fm_coef_df = pd.DataFrame(fm_results_table)
print(f'Fama-MacBeth Regression, {n_months_fm} months, {len(fm_available)} characteristics')
print(fm_coef_df.to_string(index = False))

### FM predictive portfolio (long-short and long-only)

In [ ]:

min_history = 60
fm_predictions = {}

for t_idx in range(min_history, len(fm_dates_used)):
    beta_avg = fm_betas[:t_idx].mean(axis = 0)
    pred_date = fm_dates_used[t_idx]
    m = all_months[pred_date]
    pred = beta_avg[0] + m['fm_x'] @ beta_avg[1:]
    valid = np.isfinite(pred)
    if valid.sum() < 10:
        continue
    fm_predictions[pred_date] = {
        'w': pred[valid].astype(np.float32),
        'ids': m['ids'][valid],
        'r': m['r'][valid].astype(np.float32),
    }

print(f'FM predictive portfolio months out-of-sample, {len(fm_predictions)}')

fm_corrs = []
for p in fm_predictions.values():
    if len(p['w']) < 10:
        continue
    c, _ = spearmanr(p['w'], p['r'])
    if not np.isnan(c):
        fm_corrs.append(float(c))
fm_rc = float(np.mean(fm_corrs)) if fm_corrs else 0.0
print(f'FM rank correlation, {fm_rc:.4f}')

keys_fm = sorted(fm_predictions.keys())
rset_fm = set(keys_fm[::rebalance_freq])

fm_ls_rets, fm_ls_rb_indices = [], []
fm_lo_rets, fm_lo_rb_indices = [], []
li_ids, si_ids = set(), set()
prev_li, prev_si = set(), set()

for eom in keys_fm:
    p = fm_predictions[eom]
    w = p['w']
    ids = p['ids']
    r = p['r']

    ls_tcv = 0.0
    lo_tcv = 0.0

    if eom in rset_fm:
        fm_ls_rb_indices.append(len(fm_ls_rets))
        fm_lo_rb_indices.append(len(fm_lo_rets))
        nq = max(1, int(len(w) * 0.20))
        so = np.argsort(w)
        li_ids = set(ids[so[::-1][:nq]].tolist())
        si_ids = set(ids[so[:nq]].tolist())

        ls_to = (
            len(li_ids - prev_li) + len(prev_li - li_ids)
            + len(si_ids - prev_si) + len(prev_si - si_ids)
        ) / max(nq, 1)
        ls_tcv = ls_to * tc_bps / 10000.0
        lo_to = (
            len(li_ids - prev_li) + len(prev_li - li_ids)
        ) / max(nq, 1)
        lo_tcv = lo_to * tc_bps / 10000.0

        prev_li = li_ids
        prev_si = si_ids

    if not li_ids:
        continue
    il = ids.tolist()
    lr = r[np.array([i in li_ids for i in il])]
    sr = r[np.array([i in si_ids for i in il])]
    lr_mean = float(lr.mean()) if len(lr) > 0 else 0.0
    sr_mean = float(sr.mean()) if len(sr) > 0 else 0.0
    fm_ls_rets.append(lr_mean - sr_mean - ls_tcv)
    fm_lo_rets.append(lr_mean - lo_tcv)

fm_ls_rets = np.array(fm_ls_rets)
fm_lo_rets = np.array(fm_lo_rets)
fm_ls_scaled = apply_vol_target(fm_ls_rets, fm_ls_rb_indices, target_vol, vol_lookback, max_leverage_ls)
fm_lo_scaled = apply_vol_target(fm_lo_rets, fm_lo_rb_indices, target_vol, vol_lookback, max_leverage_lo)

fm_ls_raw_m = portfolio_metrics(fm_ls_rets)
fm_ls_scaled_m = portfolio_metrics(fm_ls_scaled)
fm_lo_raw_m = portfolio_metrics(fm_lo_rets)
fm_lo_scaled_m = portfolio_metrics(fm_lo_scaled)

print(f'FM long-short raw, sharpe = {fm_ls_raw_m["sharpe"]:.4f}')
print(f'FM long-short scaled, sharpe = {fm_ls_scaled_m["sharpe"]:.4f}, ann_ret = {fm_ls_scaled_m["ann_ret"] * 100:.2f}%, ann_vol = {fm_ls_scaled_m["ann_vol"] * 100:.2f}%')
print(f'FM long-only raw, sharpe = {fm_lo_raw_m["sharpe"]:.4f}')
print(f'FM long-only scaled, sharpe = {fm_lo_scaled_m["sharpe"]:.4f}, ann_ret = {fm_lo_scaled_m["ann_ret"] * 100:.2f}%, ann_vol = {fm_lo_scaled_m["ann_vol"] * 100:.2f}%')

### Save Results

In [ ]:
for fname, fr in factor_results.items():
    np.save(results_dir / f'em_{fname}_returns_ls_raw.npy', fr['returns_ls_raw'])
    np.save(results_dir / f'em_{fname}_returns_ls_scaled.npy', fr['returns_ls_scaled'])
    np.save(results_dir / f'em_{fname}_returns_lo_raw.npy', fr['returns_lo_raw'])
    np.save(results_dir / f'em_{fname}_returns_lo_scaled.npy', fr['returns_lo_scaled'])

np.save(results_dir / 'em_market_returns.npy', market_rets)
np.save(results_dir / 'em_fm_returns_ls_raw.npy', fm_ls_rets)
np.save(results_dir / 'em_fm_returns_ls_scaled.npy', fm_ls_scaled)
np.save(results_dir / 'em_fm_returns_lo_raw.npy', fm_lo_rets)
np.save(results_dir / 'em_fm_returns_lo_scaled.npy', fm_lo_scaled)

fm_beta_df = pd.DataFrame(
    fm_betas, columns = ['intercept'] + fm_available,
    index = pd.DatetimeIndex(fm_dates_used),
)
fm_beta_df.to_csv(results_dir / 'em_fm_monthly_betas.csv')

summary = {
    'universe': 'EM',
    'fm_characteristics': fm_available,
    'market_long_only': mkt_m,
    'factors': {
        f: {
            'long_short_scaled': fr['metrics_ls_scaled'],
            'long_short_raw': fr['metrics_ls_raw'],
            'long_only_scaled': fr['metrics_lo_scaled'],
            'long_only_raw': fr['metrics_lo_raw'],
        }
        for f, fr in factor_results.items()
    },
    'fm_regression': {
        'n_months': n_months_fm, 'n_chars': len(fm_available),
        'characteristics': fm_available, 'coefficients': fm_results_table,
    },
    'fm_portfolio': {
        'long_short_raw': fm_ls_raw_m,
        'long_short_scaled': fm_ls_scaled_m,
        'long_only_raw': fm_lo_raw_m,
        'long_only_scaled': fm_lo_scaled_m,
        'rank_corr': fm_rc, 'n_oos_months': len(fm_predictions),
    },
}
with open(results_dir / 'em_ff_summary.json', 'w') as fh:
    json.dump(summary, fh, indent = 2, default = float)


### Summary

In [ ]:
summary_rows = []

summary_rows.append({
    'strategy': 'market_value_weighted', 'portfolio': 'long_only',
    'sharpe': round(mkt_m['sharpe'], 4),
    'se': np.nan,
    'ann_ret': round(mkt_m['ann_ret'] * 100, 2),
    'ann_vol': round(mkt_m['ann_vol'] * 100, 2),
    'max_dd': np.nan,
    'n_obs': mkt_m['n_obs'],
})

for fname in ['value', 'momentum', 'profitability', 'investment', 'size']:
    if fname not in factor_results:
        continue
    for label, key in [('long_short', 'metrics_ls_scaled'), ('long_only', 'metrics_lo_scaled')]:
        m = factor_results[fname][key]
        summary_rows.append({
            'strategy': fname, 'portfolio': label,
            'sharpe': round(m['sharpe'], 4),
            'se': round(m['se_sharpe'], 4),
            'ann_ret': round(m['ann_ret'] * 100, 2),
            'ann_vol': round(m['ann_vol'] * 100, 2),
            'max_dd': round(m['max_dd'] * 100, 2),
            'n_obs': m['n_obs'],
        })

for label, m in [('long_short', fm_ls_scaled_m), ('long_only', fm_lo_scaled_m)]:
    summary_rows.append({
        'strategy': 'fm_regression', 'portfolio': label,
        'sharpe': round(m['sharpe'], 4),
        'se': round(m['se_sharpe'], 4),
        'ann_ret': round(m['ann_ret'] * 100, 2),
        'ann_vol': round(m['ann_vol'] * 100, 2),
        'max_dd': round(m['max_dd'] * 100, 2),
        'n_obs': m['n_obs'],
    })

summary_table = pd.DataFrame(summary_rows)
print('Fama-French Benchmark, EM Universe, vol-targeted')
print(summary_table.to_string(index = False))
print(f'FM rank correlation, {fm_rc:.4f}')

In [ ]:
matplotlib.rcParams['font.size'] = 12
fig, axes = plt.subplots(2, 2, figsize = (14, 10))
fig.suptitle('Fama-French Five-Factor Benchmark, EM Universe', fontsize = 14)

# sharpe ratios by strategy, side-by-side long-short and long-only
all_names = ['Market']
all_sharpes_ls = [mkt_m['sharpe']]
for fname in ['value', 'momentum', 'profitability', 'investment', 'size']:
    if fname in factor_results:
        all_names.append(fname.title())
        all_sharpes_ls.append(factor_results[fname]['metrics_ls_scaled']['sharpe'])
all_names.append('FM Reg')
all_sharpes_ls.append(fm_ls_scaled_m['sharpe'])

all_sharpes_lo = [mkt_m['sharpe']]
for fname in ['value', 'momentum', 'profitability', 'investment', 'size']:
    if fname in factor_results:
        all_sharpes_lo.append(factor_results[fname]['metrics_lo_scaled']['sharpe'])
all_sharpes_lo.append(fm_lo_scaled_m['sharpe'])

x_pos = np.arange(len(all_names))
axes[0, 0].bar(x_pos - 0.2, all_sharpes_ls, width = 0.4, label = 'long_short', color = 'steelblue')
axes[0, 0].bar(x_pos + 0.2, all_sharpes_lo, width = 0.4, label = 'long_only', color = 'darkorange')
axes[0, 0].set_xticks(x_pos)
axes[0, 0].set_xticklabels(all_names, rotation = 35, ha = 'right', fontsize = 9)
axes[0, 0].set_title('Sharpe Ratios by Strategy (vol-targeted)')
axes[0, 0].legend()
axes[0, 0].grid(axis = 'y', alpha = 0.3)

# fm regression cumulative wealth, both portfolios plus market
axes[0, 1].plot(np.cumprod(1.0 + fm_ls_scaled), label = 'FM long_short', color = 'steelblue')
axes[0, 1].plot(np.cumprod(1.0 + fm_lo_scaled), label = 'FM long_only', color = 'darkorange')
axes[0, 1].plot(np.cumprod(1.0 + market_rets), label = 'Market', color = 'black', linestyle = '--')
axes[0, 1].set_title('FM Cumulative Wealth (vol-targeted)')
axes[0, 1].legend()
axes[0, 1].grid(alpha = 0.3)

# fm regression t-statistics
fm_names = ['intercept'] + fm_available
x2 = np.arange(len(fm_names))
bar_colors = ['steelblue' if abs(fm_tstat[i]) > 1.96 else 'lightgray' for i in range(len(fm_names))]
axes[1, 0].barh(x2, fm_tstat, color = bar_colors)
axes[1, 0].axvline(1.96, color = 'red', linestyle = '--', alpha = 0.5)
axes[1, 0].axvline(-1.96, color = 'red', linestyle = '--', alpha = 0.5)
axes[1, 0].set_yticks(x2)
axes[1, 0].set_yticklabels(fm_names, fontsize = 9)
axes[1, 0].set_title('FM Regression t-statistics')
axes[1, 0].grid(axis = 'x', alpha = 0.3)

# factor cumulative wealth long-only
for fname in ['value', 'momentum', 'profitability', 'investment', 'size']:
    if fname in factor_results:
        axes[1, 1].plot(
            np.cumprod(1.0 + factor_results[fname]['returns_lo_scaled']),
            label = fname.title(),
        )
axes[1, 1].set_title('Factor Long-Only Cumulative Wealth')
axes[1, 1].legend(fontsize = 8)
axes[1, 1].grid(alpha = 0.3)

plt.tight_layout()
plt.savefig(results_dir / 'em_ff_benchmark.png', dpi = 150, bbox_inches = 'tight')
plt.show()
print(f'plot saved,{results_dir / "em_ff_benchmark.png"}')